# Actividad grupal clase 6 — Series temporales clásicas

**BCD5105 · Modelado matemático · III Cuatrimestre 2026 · Lead University**

**Integrantes del grupo:** _(listar aquí a los integrantes)_

> Este notebook usa el **entorno de R** (Colab: *Entorno de ejecución → Cambiar tipo de entorno → R*). Las celdas de código llevan como comentario la salida que debe aparecer con `set.seed(2026)` y las celdas en orden.

Contenido:
1. Miniexploración a mano: diferencias de una serie trimestral y lectura de tres ACF
2. La serie y su receta (tsibble, estacionalidad, STL)
3. ARIMA automático
4. ¿Le gana al ingenuo? (SARIMA vs. SNAIVE con MASE)
5. Reflexión

In [ ]:
# Tarda varios minutos en Colab: correrla apenas se abra el notebook, mientras se trabaja la parte 1.
# Si se reinicia el entorno, hay que volver a instalar.
install.packages("fpp3")

---
## Parte 1: Miniexploración a mano

### Situación A: colocaciones trimestrales de una cooperativa (millones de colones)

| Trimestre | 2024 T1 | T2 | T3 | T4 | 2025 T1 | T2 | T3 | T4 |
|---|---|---|---|---|---|---|---|---|
| $y$ | 20 | 26 | 18 | 24 | 23 | 29 | 21 | 27 |

**1. Primera diferencia** $y_t-y_{t-1}$:

| De → a | T1→T2 | T2→T3 | T3→T4 | T4→T1 | T1→T2 | T2→T3 | T3→T4 |
|---|---|---|---|---|---|---|---|
| $\Delta y$ | 26−20 = **6** | 18−26 = **−8** | 24−18 = **6** | 23−24 = **−1** | 29−23 = **6** | 21−29 = **−8** | 27−21 = **6** |

La primera diferencia es $6,\,-8,\,6,\,-1,\,6,\,-8,\,6$. **El patrón trimestral no desapareció**: sigue el vaivén "sube 6, baja 8, sube 6, y casi no cambia (−1)" que se repite cada año. La diferencia de rezago 1 elimina el nivel y reduce la tendencia, pero la estacionalidad queda.

**2. Diferencia estacional** $y_t-y_{t-4}$ (para $t=5,\dots,8$):

$$23-20=3,\qquad 29-26=3,\qquad 21-18=3,\qquad 27-24=3.$$

Queda una **constante, 3**: las colocaciones crecen **3 millones por año** en cada trimestre, sobre un patrón trimestral que es el mismo en ambos años. La diferencia estacional elimina la estacionalidad (y deja solo el crecimiento anual), lo que la hace útil para llevar la serie a algo estacionario.

In [ ]:
y <- c(20, 26, 18, 24, 23, 29, 21, 27)
diff(y)               # primera diferencia:   6 -8  6 -1  6 -8  6
diff(y, lag = 4)      # diferencia estacional: 3  3  3  3

### Situación B: tres ACF descritas en palabras

| ACF | Qué muestra | Interpretación | Serie |
|---|---|---|---|
| (i) | Todas las barras dentro de la franja azul | No hay autocorrelación: **ruido blanco** (impredecible a partir del pasado) | **Casos semanales de un evento al azar** |
| (ii) | Barras altas que bajan muy despacio, rezago tras rezago | Serie con **tendencia** / no estacionaria: cada valor se parece mucho al anterior | **Población del país año a año** |
| (iii) | Barras altas en los rezagos 12, 24 y 36 y negativas cerca de 6, 18 y 30 | **Estacionalidad de período 12** (a 6 meses el patrón está en la fase opuesta) | **Ventas mensuales de helados** |

**4. ¿Cuál necesita una diferencia estacional?** La **(iii)**, ventas de helados: la estacionalidad anual (rezago 12) se elimina con $y_t-y_{t-12}$. La (ii) necesitaría una diferencia **ordinaria** ($y_t-y_{t-1}$) por la tendencia, y la (i) ninguna.

Para comprobarlo se simulan tres series con esos comportamientos y se dibuja su ACF:

In [ ]:
set.seed(1)
t <- 1:120
ruido      <- rnorm(120)                                   # (i)   evento al azar
poblacion  <- cumsum(rnorm(120, mean = 1))                 # (ii)  tendencia (camino aleatorio con deriva)
helados    <- 10 + 5 * sin(2 * pi * t / 12) + rnorm(120)   # (iii) estacionalidad de período 12

par(mfrow = c(1, 3))
acf(ruido,     lag.max = 36, main = "(i) ruido blanco")
acf(poblacion, lag.max = 36, main = "(ii) tendencia")
acf(helados,   lag.max = 36, main = "(iii) estacionalidad (12)")
par(mfrow = c(1, 1))

---
## Parte 2: La serie y su receta

Ocho años de demanda eléctrica mensual (GWh): tendencia creciente (2.5 GWh por mes) y patrón estacional multiplicativo: más consumo en la época seca y caliente, menos a final de la época lluviosa.

In [ ]:
library(fpp3)
set.seed(2026)
n <- 96; t <- 1:n
patron <- c(0.98, 0.97, 1.04, 1.06, 1.05, 1.00,
            0.99, 1.00, 0.97, 0.98, 0.97, 0.99)
demanda <- (900 + 2.5 * t) * patron[((t - 1) %% 12) + 1] *
  (1 + rnorm(n, 0, 0.012))
energia <- tsibble(Mes = make_yearmonth(2018, 1) + 0:(n - 1),
                   GWh = round(demanda, 1), index = Mes)

energia |> autoplot(GWh)
energia |> gg_season(GWh)
energia |> model(STL(GWh ~ season(window = "periodic"), robust = TRUE)) |>
  components() |> autoplot()
energia |> features(GWh, feat_stl)
# trend_strength 0.981, seasonal_strength_year 0.917
# seasonal_peak_year 4 (abril), seasonal_trough_year 9 (setiembre)

**1. La serie.** `autoplot` muestra una tendencia creciente y casi lineal con oscilaciones anuales regulares.

**2. `gg_season`.** El mes de **mayor demanda es abril** y el de **menor demanda es setiembre** (coincide con `seasonal_peak_year = 4` y `seasonal_trough_year = 9`).

**3. STL y fuerza de los componentes.** `trend_strength = 0.981` y `seasonal_strength_year = 0.917`, en una escala de 0 a 1: tanto la tendencia como la estacionalidad son **muy fuertes** y explican casi toda la variación; lo que queda en el residuo es poco.

### Pregunta para discutir

- **Explicación física del patrón.** En un país tropical como Costa Rica el año se divide en época **seca** (diciembre–abril) y **lluviosa** (mayo–noviembre). El pico de **abril** es el final de la estación seca y la más calurosa: más uso de aire acondicionado y ventiladores, más bombeo de agua y riego. El valle de **setiembre** cae en lo más lluvioso del año, con temperaturas más frescas, menos consumo de refrigeración y riego. Febrero, con 28 días, tiene un factor un poco más bajo (0.97) que enero.
- **¿Aditiva o multiplicativa?** Por cómo se simuló, es **multiplicativa**: la demanda es `(900 + 2.5 t) × patrón`, de modo que la amplitud de las oscilaciones crece con el nivel de la serie (entre el primer y el último año la diferencia entre el mes más alto y el más bajo pasa de unos 87 a unos 106 GWh, ≈ +22 %). Como el nivel solo sube ~25 % en ocho años, a simple vista la diferencia con una serie aditiva es sutil; si se quisiera estabilizar la amplitud, se modelaría `log(GWh)`.

---
## Parte 3: ARIMA automático

**1. Predicción antes de ajustar.** KPSS (`unitroot_nsdiffs`) debería recomendar **1 diferencia estacional** (la estacionalidad es muy fuerte). Tras diferenciar estacionalmente $y_t-y_{t-12}$, queda aproximadamente una constante (el crecimiento anual) más ruido, así que esperamos un modelo **sin términos AR/MA ordinarios**, con una diferencia estacional, un término de **media móvil estacional** y una **constante (deriva)**: ARIMA(0,0,0)(0,1,1)[12] con deriva.

In [ ]:
energia |> features(GWh, unitroot_nsdiffs)     # 1
fit <- energia |> model(ARIMA(GWh))
report(fit)
# ARIMA(0,0,0)(0,1,1)[12] w/ drift
#          sma1 constant
#       -0.6802  30.4857
fit |> gg_tsresiduals()
augment(fit) |> features(.innov, ljung_box, lag = 24, dof = 1)
# lb_stat 16.7   lb_pvalue 0.824

**2. Modelo elegido.** `ARIMA(0,0,0)(0,1,1)[12] w/ drift`: una diferencia estacional con deriva, la misma estructura que el turismo de la clase, y un término de media móvil estacional `sma1 = −0.68`. KPSS recomendó 1 diferencia estacional, tal como se predijo.

**3. La constante.** El modelo es $(1-B^{12})y_t = c+\varepsilon_t+\Theta\,\varepsilon_{t-12}$, así que la constante $c\approx30.5$ es lo que crece la demanda **de un año al siguiente**: la tendencia simulada sube 2.5 GWh por mes, es decir, $12\times2.5=30$ GWh por año. Coincide con la constante estimada (30.49).

**4. Residuos.** `gg_tsresiduals()` no debería mostrar barras fuera de la franja en el ACF, y la prueba de Ljung-Box (24 rezagos, `dof = 1` por el único parámetro ARMA estimado, `sma1`) da `lb_pvalue = 0.824`: no se rechaza que los residuos sean ruido blanco, es decir, no queda estructura de autocorrelación sin modelar y el modelo es adecuado.

---
## Parte 4: ¿Le gana al ingenuo?

Se entrena con 2018–2024 y se deja 2025 como prueba. SNAIVE repite cada mes el valor del mismo mes del año anterior.

In [ ]:
entreno <- energia |> filter(Mes < make_yearmonth(2025, 1))
modelos <- entreno |> model(snaive = SNAIVE(GWh), sarima = ARIMA(GWh))

modelos |> forecast(h = 12) |> accuracy(energia) |>
  select(.model, RMSE, MAPE, MASE) |> arrange(MASE)
#   .model  RMSE  MAPE  MASE
# 1 sarima  15.5  1.09 0.393
# 2 snaive  34.0  2.72 0.982

# modelo que eligió el algoritmo con solo el entrenamiento
modelos |> select(sarima) |> report()
# ARIMA(0,0,0)(2,1,1)[12] w/ drift   (puede variar con la versión de fable)

modelos |> forecast(h = 12) |>
  autoplot(energia |> filter(Mes >= make_yearmonth(2023, 1)), level = NULL)

**Resultado.** El SARIMA comete **menos de la mitad** del error del ingenuo: RMSE 15.5 contra 34.0 GWh y MASE **0.39 contra 0.98**. Con solo el entrenamiento, el algoritmo eligió un modelo algo distinto, ARIMA(0,0,0)(2,1,1)[12] con deriva: con menos datos, la búsqueda automática puede cambiar de modelo (lo que importa es comparar el MASE).

### Preguntas

- **¿Por qué SNAIVE queda tan cerca de 1?** El MASE divide el error en la prueba por el error del **propio ingenuo estacional dentro del entrenamiento**. Si el comportamiento de la serie en la prueba se parece al del entrenamiento, SNAIVE da un MASE ≈ 1 por construcción: es la vara con que se mide. Un MASE < 1 quiere decir "mejor que el ingenuo del entrenamiento", y 0.98 es "igual de bueno que él".
- **¿Qué parte de la serie no puede capturar SNAIVE?** La **tendencia**. Repite el valor del año anterior, así que se queda corto en la subida de ~30 GWh por año, y ese sesgo es la mayor parte de su error: $\sqrt{30^2+(\sqrt2\cdot 13)^2}\approx 35$, compatible con su RMSE de 34. Además hereda el ruido del año pasado: usa **un solo dato ruidoso** como pronóstico, con lo que el ruido se cuenta dos veces. SARIMA suma la deriva (+30) y promedia el patrón estacional con la media móvil; su RMSE de 15.5 está cerca del ruido irreducible (≈ 1.2 % de ~1100 GWh ≈ 13 GWh).

---
## Parte 5: Reflexión

**Observación matemática.** La diferencia estacional de la parte 1 reveló lo que **no se veía en la serie original**: que las colocaciones crecen exactamente **3 millones por año** en todos los trimestres y que el patrón trimestral es fijo. En la serie original, la tendencia y la estacionalidad están mezcladas; la primera diferencia no logra separarlas (el patrón trimestral sobrevive: 6, −8, 6, −1, …) y la diferencia estacional sí. Es la misma idea que en el ARIMA de la energía: la constante de 30 GWh es el crecimiento anual que queda tras diferenciar.

**Observación técnica.** No se debe validar un pronóstico con un **K-fold barajado** porque las observaciones tienen orden temporal y dependen unas de otras: al barajar, el modelo se entrena con datos **posteriores** a los que luego "pronostica" (fuga del futuro), y los vecinos inmediatos de cada punto de validación están en el entrenamiento, así que el error resulta artificialmente bajo y no refleja la situación real de pronosticar el futuro desconocido. La validación debe respetar el tiempo: entrenar con el pasado y probar con lo que viene después (como el año de prueba de la parte 4), o con ventana expansiva (`stretch_tsibble()`) para varios orígenes.

**Pregunta abierta.** ¿La serie de nuestro proyecto tiene estacionalidad? ¿Con qué período? _(completar según el carril del grupo: frecuencia de los datos —diaria, semanal, mensual—, período esperado —7, 12, 52 …—, se confirma con `gg_season`, el ACF y `feat_stl`; si es mensual y anual, período 12 como aquí.)_